# 4.1 Engagement prevalence in the experimental sample

**Author:** Kristina Aleksandrovna Pedersen

This code calculates bookmarks, likes, and shares per 100 views for the political videos, entertainment videos, and all 400 videos used in the simulated feed.

In [1]:
#Load relevant modules
import pandas as pd
from glob import glob

### Load the final survey sample

In [2]:
#Load the exact 400 videos included in the simulated feed
sample = pd.read_excel("../../SURVEY_FOLDER/video_bundle_survey_overview.xlsx")
sample.tiktok_id = sample.tiktok_id.astype(str)
print(sample.shape)
sample.content.value_counts()

(400, 7)


content
political        219
entertainment    181
Name: count, dtype: int64

### Add the video engagement metadata

In [3]:
#Political-video metadata
metadata_columns = ['video_id', 'saved', 'likes', 'shares', 'views']
political_files = (glob("../../video_metadata_clean/*/*.pkl") +
                   glob("../../video_metadata_new/*/*.pkl"))

dfs = []
for file in political_files:
    df = pd.read_pickle(file)
    if all(column in df.columns for column in metadata_columns):
        dfs.append(df[metadata_columns])

political_metadata = pd.concat(dfs, ignore_index = True)
political_metadata.video_id = political_metadata.video_id.astype(str)

#The newer metadata files are last in the list and are retained when available
political_metadata.drop_duplicates(subset = 'video_id', keep = 'last', inplace = True)

In [4]:
#Entertainment-video metadata
entertainment_metadata = pd.read_pickle("../../entertainment/ent_video_metadata_clean.pkl")
entertainment_metadata.video_id = entertainment_metadata.video_id.astype(str)
entertainment_metadata = entertainment_metadata[metadata_columns]
entertainment_metadata.drop_duplicates(subset = 'video_id', keep = 'last', inplace = True)

In [5]:
#Merge each video type with the corresponding metadata
political = sample[sample.content == 'political'][['tiktok_id', 'content']].merge(
    political_metadata, left_on = 'tiktok_id', right_on = 'video_id', how = 'left')

entertainment = sample[sample.content == 'entertainment'][['tiktok_id', 'content']].merge(
    entertainment_metadata, left_on = 'tiktok_id', right_on = 'video_id', how = 'left')

videos = pd.concat([political, entertainment], ignore_index = True)

print("Political videos with metadata:", political.views.notnull().sum(), "of", political.shape[0])
print("Entertainment videos with metadata:", entertainment.views.notnull().sum(), "of", entertainment.shape[0])

Political videos with metadata: 219 of 219
Entertainment videos with metadata: 181 of 181


### Engagement per 100 views

The calculation is the total number of each engagement type divided by total views, multiplied by 100.

In [6]:
#Make sure the engagement columns are numeric
engagement_columns = ['saved', 'likes', 'shares', 'views']
videos[engagement_columns] = videos[engagement_columns].apply(pd.to_numeric, errors = 'coerce')

#Sum engagements and views separately for political and entertainment videos
summary = videos.groupby('content').agg(
    Videos = ('tiktok_id', 'count'),
    Bookmarks = ('saved', 'sum'),
    Likes = ('likes', 'sum'),
    Shares = ('shares', 'sum'),
    Views = ('views', 'sum')
)

#Add all videos as a third group
summary.loc['all'] = [
    videos.shape[0], videos.saved.sum(), videos.likes.sum(),
    videos.shares.sum(), videos.views.sum()
]

#Calculate engagements per 100 views
summary['Bookmarks per 100 views'] = 100 * summary.Bookmarks / summary.Views
summary['Likes per 100 views'] = 100 * summary.Likes / summary.Views
summary['Shares per 100 views'] = 100 * summary.Shares / summary.Views

summary.index = summary.index.map({
    'political': 'Political',
    'entertainment': 'Entertainment',
    'all': 'All'
})
summary.index.name = 'Content'
summary.round(3)

,Videos,Bookmarks,Likes,Shares,Views,Bookmarks per 100 views,Likes per 100 views,Shares per 100 views
Content,,,,,,,,
Entertainment,181.0,15723972.0,225248001.0,24041943.0,2.280516e+09,0.689,9.877,1.054
Political,219.0,54513.0,956081.0,52596.0,1.306216e+07,0.417,7.319,0.403
All,400.0,15778485.0,226204082.0,24094539.0,2.293578e+09,0.688,9.862,1.051
